# Trung bình 4 lớp ẩn cuối — CafeBERT, bài category (nhúng pooled, train + dev)

Thay vector lớp cuối bằng trung bình 4 lớp ẩn cuối (`output_hidden_states=True`) rồi mean pooling
có mặt nạ như cũ. Shape đầu ra `[n, 1024]`; `max_len=512`; chỉ nhúng `train` và `dev` — **không đụng `test`** (Rule 5).
CafeBERT là XLM-R-large (1024 chiều), 25 trạng thái ẩn, `output_hidden_states` giữ cả 25 activation nên tốn VRAM hơn; chạy trên T4 ước tính ≈ 60–70 phút (chưa đo).

Mốc để so (dev, lớp cuối, 5 seed): CE 0,6037 ± 0,0037; cw effective β=0,999 + ls 0,05 0,6150 ± 0,0027.

**Trước khi chạy:** Runtime → Change runtime type → **GPU**. Cần trên Drive của tài khoản này:
`MyDrive/vietjobs_splits/` (train.csv dev.csv test.csv manifest.json).
Notebook này **chỉ nhúng**; huấn luyện Dense chạy trên Mac, `--device cpu` (Rule 13 mục 5).

In [ ]:
!nvidia-smi | head -15
!df -h /content | tail -1

In [ ]:
import os
if not os.path.isdir('vietjobs'):
    !git clone -q -b t8-rnn-colab https://github.com/boy2407/vietjobs.git
%cd vietjobs
# KHÔNG cài lại torch — giữ bản CUDA sẵn có của Colab.
!pip install -q "transformers>=4.38,<4.47" "pyarrow>=15.0" "scikit-learn>=1.5" \
    "scipy>=1.13" "joblib>=1.4" "underthesea>=6.8" sentencepiece protobuf pytest
!pip install -q -e . --no-deps
%env PYTHONPATH=src

## Kiểm mã có `--layers`

Nhánh `t8-rnn-colab` đã có `--layers` (commit `8b80f11`), nên không cần chép mã đè lên bản clone.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
assert '--layers' in open('src/vietjobs/dl/encode.py').read(), 'encode.py chưa có --layers'
print('encode.py có --layers')


## Khôi phục splits

In [ ]:
!python notebooks/colab_setup.py --family raw --no-train-token --skip-tests

In [ ]:
!python -m pytest -q tests/test_dl_encode.py

## Nhúng `dev` trước, rồi `train`

Hai lệnh riêng: cache pooled chỉ được ghi khi xong cả split, nên bị ngắt giữa `train` chỉ mất lượt `train`.
Hết VRAM thì giảm `--batch` (32 → 16).

In [ ]:
!python -m vietjobs.dl.encode --task category --splits dev --encoder cafebert --max-len 512 --layers 4 --device cuda --batch 32
!cat artifacts/embeddings/*/cafebert-dev-raw-len512-last4.json

In [ ]:
!python -m vietjobs.dl.encode --task category --splits train --encoder cafebert --max-len 512 --layers 4 --device cuda --batch 32

## Kiểm shape và sao lưu lên Drive

In [ ]:
import glob, numpy as np, json
for split, n in (('train', 34354), ('dev', 3812)):
    p = glob.glob(f'artifacts/embeddings/*/cafebert-{split}-raw-len512-last4.npy')[0]
    a = np.load(p, mmap_mode='r')
    assert a.shape == (n, 1024), (p, a.shape)
    assert np.isfinite(a).all()
    print(p, a.shape, a.dtype, json.load(open(p.replace('.npy', '.json')))['layers'], 'lớp')

In [ ]:
!mkdir -p /content/drive/MyDrive/vietjobs_cache_last4
!cp artifacts/embeddings/*/cafebert-*-raw-len512-last4.* /content/drive/MyDrive/vietjobs_cache_last4/
!ls -l /content/drive/MyDrive/vietjobs_cache_last4/
!md5sum /content/drive/MyDrive/vietjobs_cache_last4/cafebert-*-last4.npy

## Mang về máy chính

Tải 4 tệp trong `MyDrive/vietjobs_cache_last4/` (`cafebert-train-raw-len512-last4.{npy,json}`, `cafebert-dev-raw-len512-last4.{npy,json}`),
đặt vào `artifacts/embeddings/<SPLIT_TAG>/` trên Mac, so `md5sum`, rồi huấn luyện:

```bash
PYTHONPATH=src .venv-dl/bin/python -m vietjobs.dl.train_dl --task category --encoder cafebert --max-len 512 --layers 4 --seed 42 --run-id dl-cat-cafe-last4-ce-s42
```